Estoy trabajando en Spark.

Tengo un dataframe A con una columna "id_sample" y B filas.

Tengo otro dataframe B con varias columnas, entre ellas "id_sample" con N filas.

N >> B y B= 0.05*N

¿Que es mas eficiente en Spark, considerando que ambos dataframes estan distribuidos en varias particiones?

- Hacer un left join de ambos dataframes a traves de la columna id_sample.

- Hacer broadcast de los B id_sample del dataframe A y hacer left join con dataframe B de estos valores broadcast.

El Broadcast Join (hacer broadcast del DataFrame A) es drásticamente más eficiente.Al ser $B$ una fracción pequeña ($5\%$ de $N$) y contener únicamente la columna id_sample, el tamaño en megabytes del DataFrame A es mínimo.

¿Por qué el Broadcast Join es superior?

**Elimina el Shuffle de $N$ filas (DataFrame B):**

- En un Join estándar (SortMergeJoin / ShuffleHashJoin), Spark debe redistribuir por red (shuffle) ambos DataFrames a través del cluster para que las filas con el mismo id_sample caigan en el mismo nodo. Mover $N$ filas con todas sus columnas por la red es la operación más costosa en tiempo de ejecución, I/O y memoria.
- En un Broadcast Join, el DataFrame B se queda en las particiones donde ya reside (cero transferencia por red de $N$).

**Costo de red mínimo y focalizado:**

- El Driver recolecta las $B$ claves de A y las envía comprimidas a la memoria de cada Executor. Como A solo tiene la columna id_sample, esta transferencia es ligera y toma milisegundos.

**Búsqueda en memoria $O(1)$:**

Cada Executor convierte el DataFrame A transmitido en una tabla Hash en memoria local y evalúa sus particiones locales de B de forma directa y paralela.